<a href="https://colab.research.google.com/github/va234-a11y/Project-2-TCGA-p53-Capstone-Project/blob/main/03_extension.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 Notebook 03 — Extension

**Worth 25 marks. Choose ONE.**

---

> **Read this first.** The extension is not required to do well. A
> complete, careful core — Notebooks 01 and 02, properly written and
> properly interpreted — is first-class work and scores around 70. This
> notebook is how you move from 70 to 85.
>
> **One extension. Not two, not all three.** Doing one well beats doing
> three badly, and there are no marks for volume.

All three use the same expression file as Notebook 02, and **none of
them need your earlier answers to be correct** — a working
`compare_groups` is given to you below.

## Every option has the same basic deliverables

Whichever option you choose, you must provide:

1. the named function, following the contract given;
2. the supplied structural checks passing;
3. **at least two useful checks of your own** — choose properties that
   would catch a plausible mistake, not merely `result is not None`;
4. a short interpretation explaining what the extension changes about
   your Notebook 02 conclusion.

The supplied checks are only a starting point. Choosing what else needs
testing is part of the extension.

> ### ⚠️ Run the setup, then only your chosen option
>
> Run the setup cells at the top first. After that, work through **Option
> A, B or C** — not all three. The untouched options still contain
> `NotImplementedError`, so **Restart and run all is not appropriate for
> this notebook** until you remove or skip the two options you did not
> choose.
>
> If you edit a function, re-run its cell before running its checks.

## Run this first

> ✏️ **Before you run the cell below, add your Sussex username.**
>
> Scroll to the **bottom** of the cell — **line 86 of 93** —
> and type your Sussex username (the start of your Sussex email, the
> one you log in with) between the quotes:
>
> ```python
> STUDENT_ID = "ab123"   # <-- your Sussex username
> ```
>
> Then run the cell (**Shift + Enter**). It prints your gene and cancer
> type. Change nothing else in that cell.
>
> 💡 To see line numbers in Colab: **Tools → Settings → Editor → Show
> line numbers**.

In [ ]:
# Run this first. It downloads your data and sets up the self-checks.
import hashlib
import json
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu

DATA_URL = "https://raw.githubusercontent.com/HocheggerLab/tcga-capstone-2026/main/data/"

def fetch(filename):
    """Return a local data path, downloading it in Colab if needed."""
    repo_path = Path("data") / filename
    working_path = Path(filename)

    if repo_path.exists():                   # running from the repo
        return repo_path
    if not working_path.exists():            # running in Colab
        urllib.request.urlretrieve(DATA_URL + filename, working_path)
    return working_path


def my_variant(student_id):
    """Look up your gene and cancer type from your Sussex username."""
    number = str(student_id).strip().lower()
    if not number:
        raise ValueError("Type your Sussex username into "
                         "STUDENT_ID, then run this cell again.")
    key = hashlib.pbkdf2_hmac("sha256", number.encode(), b'tcga-capstone-2026',
                              2_000_000).hex()[:16]
    lookup = json.loads(Path(fetch("variant_lookup.json")).read_text())
    if key not in lookup:
        raise ValueError(f"{number} is not on the class list. Check it "
                         "against Canvas; if it is right, email the convenor.")
    gene, cancer_type = lookup[key]
    print(f"{number}: your gene is {gene}, your cancer type is {cancer_type}")
    return gene, cancer_type


def run_checks(checks):
    """Run a dict of {description: test} and report what passed.

    Each test is a zero-argument function returning True or False. A
    test that raises is reported rather than stopping the notebook, so
    you always see the full picture.
    """
    passed = 0
    for description, test in checks.items():
        try:
            ok = bool(test())
        except Exception as exc:
            print(f"\u274c {description}")
            print(f"     raised {type(exc).__name__}: {exc}")
            continue
        print(f"{'\u2705' if ok else '\u274c'} {description}")
        passed += ok
    print()
    print(f"{passed} of {len(checks)} checks passed")
    if passed < len(checks):
        print("Keep going - fix the crosses and run this cell again.")


# GIVEN - a working compare_groups, so the extension does not depend on
# whether yours came out right in Notebook 02. Use it.
def compare_groups(expr, group_col, group_a, group_b):
    """Compare expression between two groups with a Mann-Whitney U test."""
    a = expr.loc[expr[group_col] == group_a, "log2_tpm"].dropna()
    b = expr.loc[expr[group_col] == group_b, "log2_tpm"].dropna()
    if a.empty or b.empty:
        raise ValueError(
            f"empty group: {group_a!r} has {len(a)} rows, "
            f"{group_b!r} has {len(b)}"
        )
    p_value = float(mannwhitneyu(a, b, alternative="two-sided").pvalue)
    return {
        "n_a": int(len(a)), "n_b": int(len(b)),
        "median_a": float(a.median()), "median_b": float(b.median()),
        "difference": float(a.median() - b.median()),
        "p_value": p_value, "significant": bool(p_value < 0.05),
    }


STUDENT_ID = ""   # <-- your Sussex username, e.g. "ab123"

GENE, CANCER_TYPE = my_variant(STUDENT_ID)

expr = pd.read_csv(fetch(f"expression_{GENE}_{CANCER_TYPE}.csv"))
tumours = expr[(expr["cancer_type"] == CANCER_TYPE)
               & (expr["group"] == "tumour")].copy()
print(len(tumours), "tumours in", CANCER_TYPE)

---

# Option A — The tumours you called "wild-type" aren't all wild-type

**The most interesting of the three — if your cancer type has the
numbers.**

> ### ⚠️ Check this before you choose Option A
>
> This option compares TP53 wild-type tumours that have **deleted** TP53
> against those with a normal copy number. In some cancer types there
> are only a handful of deleted wild-type tumours — too few to test.
>
> **Option A is available for:** `BRCA`, `LIHC`, `LUAD`, `PRAD`, `STAD`.
>
> If your cancer type is not on that list, choose **Option B** or
> **Option C**. You are not disadvantaged — all three are marked out of
> the same 25.

## The problem with your own analysis

In Notebook 02 you split tumours into TP53-**mutant** and
TP53-**wild-type**. But look at what "wild-type" actually means there:
*no mutation was found in the TP53 gene*. That is **not** the same as
*this tumour has working p53*.

The commonest way a tumour loses p53 without a point mutation is by
**deleting the chunk of chromosome 17 that carries the gene**. There is
no mutation to find — the gene simply is not there.

So your wild-type group is **contaminated** with tumours that have no
functional p53 at all. Your control group contains things that are not
controls.

## The test

Take only the tumours with no TP53 mutation, **throw the mutation data
away entirely**, and split them instead by `tp53_cn` — the copy number
of TP53.

If deletion really does knock out p53, the deleted tumours should behave
like the mutants did: your gene should move in the same direction it
moved in Notebook 02, even though not one of these tumours carries a
mutation.

## The judgement call — and it is marked

`tp53_cn` is a **continuous** log2 ratio, not a label. Nobody has told
you where "deleted" begins. Choose a threshold and **defend it in
writing**. A student who picks a number and explains why beats one who
picks the same number silently.

Look at the distribution before you choose.

In [ ]:
# DEMO - where would you put the line?
wt = tumours[tumours["tp53_status"] == "WT"]

plt.hist(wt["tp53_cn"].dropna(), bins=50)
plt.xlabel("TP53 copy number (log2 ratio)")
plt.ylabel("tumours")
plt.title(f"TP53 copy number in {CANCER_TYPE} wild-type tumours")
plt.show()

## The contract

```python
compare_tp53_deletion(expr, cancer_type, threshold)
```

**Takes** `expr`, the tidy expression table; `cancer_type`, the cancer
type to analyse; and `threshold`, the `tp53_cn` value below which a
tumour counts as deleted.

**Returns** the **same seven keys as `compare_groups`**, plus
`"threshold"` recording the cutoff you chose. **Reuse `compare_groups`** — it is given
to you above; do not write the statistics again.

In [ ]:
def compare_tp53_deletion(expr, cancer_type, threshold):
    # TODO - your turn
    raise NotImplementedError("compare_tp53_deletion")

In [ ]:
# CHECK - structure and internal consistency
deletion = compare_tp53_deletion(expr, CANCER_TYPE, -0.3)

run_checks({
    "returns a dict":
        lambda: isinstance(deletion, dict),
    "has the seven compare_groups keys plus threshold":
        lambda: set(deletion) == {"n_a", "n_b", "median_a", "median_b",
                                  "difference", "p_value", "significant",
                                  "threshold"},
    "records the threshold you passed in":
        lambda: deletion["threshold"] == -0.3,
    "difference really is median_a minus median_b":
        lambda: abs(deletion["difference"]
                    - (deletion["median_a"] - deletion["median_b"])) < 1e-9,
    "significant agrees with p_value < 0.05":
        lambda: deletion["significant"] == (deletion["p_value"] < 0.05),
    "both groups have at least 20 tumours in them":
        lambda: deletion["n_a"] >= 20 and deletion["n_b"] >= 20,
    "mutant tumours were excluded":
        lambda: deletion["n_a"] + deletion["n_b"] <= (
            (expr["tp53_status"] == "WT")
            & (expr["cancer_type"] == CANCER_TYPE)).sum(),
})

## What to write up

1. Your threshold, and **why**.
2. The result — direction and size.
3. **How it compares to your Notebook 02 answer.** Same direction?
   Bigger or smaller?
4. The consequence: if deleted tumours really are p53-null, then your
   wild-type group in Notebook 02 was contaminated — so was your
   original effect size too big, or too small?

That last point is where the marks are. Expect the deletion effect to be
**smaller** than the mutation effect, and work out why. (Hint: how many
working copies of the gene does a tumour have left after a single-copy
deletion, and how many after a mutation plus loss of the other copy?)

In [ ]:
# YOUR CHECKS - add at least two checks that would catch a real mistake.
# Then use the remaining cells you need for your Option A analysis.
#
# Example shape only - replace these descriptions and tests:
# run_checks({
#     "my first useful property": lambda: ...,
#     "my second useful property": lambda: ...,
# })

---

# Option B — Is it really p53, or just gene dosage?

**The hardest of the three.**

You found a difference in your gene's expression between TP53-mutant and
wild-type tumours, and assumed p53 caused it. There is a duller
explanation you have not ruled out: **tumours carry different numbers of
copies of your gene**, and more copies generally means more expression.

If your gene happens to be amplified more often in TP53-mutant tumours,
you would see exactly the difference you saw, with p53 having nothing to
do with it.

`gene_cn` is the copy number of **your** gene. Use it to ask whether
copy number differs between the two TP53 groups, and whether it is
enough to explain your result.

**Either answer is a good answer.** "Copy number is balanced between the
groups, so dosage is unlikely to explain the difference" makes your
Notebook 02 conclusion much harder to argue with.

## The contract

```python
analyse_gene_dosage(expr, cancer_type)
```

**Takes** `expr`, the tidy expression table, and the cancer type to
analyse. Use tumour samples with a known `tp53_status`.

**Returns** a dict with exactly these keys:

```python
{"n_mut": int, "n_wt": int,
 "median_cn_mut": float, "median_cn_wt": float,
 "cn_difference": float,       # mutant minus wild-type
 "cn_p_value": float,          # two-sided Mann-Whitney U
 "cn_significant": bool,       # p < 0.05
 "expression_cn_r": float}     # Spearman correlation, gene_cn vs log2_tpm
```

The group comparison asks whether dosage differs between TP53 groups.
The correlation asks whether copy number and expression actually move
together. Together they tell you whether dosage is a plausible
alternative explanation; they do not prove causation.

In [ ]:
def analyse_gene_dosage(expr, cancer_type):
    # TODO - your turn
    raise NotImplementedError("analyse_gene_dosage")

In [ ]:
# CHECK - structure and internal consistency
dosage = analyse_gene_dosage(expr, CANCER_TYPE)

run_checks({
    "returns a dict with exactly the eight required keys":
        lambda: set(dosage) == {"n_mut", "n_wt", "median_cn_mut",
                                "median_cn_wt", "cn_difference",
                                "cn_p_value", "cn_significant",
                                "expression_cn_r"},
    "both TP53 groups contain tumours":
        lambda: dosage["n_mut"] > 0 and dosage["n_wt"] > 0,
    "copy-number difference is mutant minus wild-type":
        lambda: abs(dosage["cn_difference"]
                    - (dosage["median_cn_mut"]
                       - dosage["median_cn_wt"])) < 1e-9,
    "significance agrees with p < 0.05":
        lambda: dosage["cn_significant"] == (dosage["cn_p_value"] < 0.05),
    "the correlation is a valid coefficient":
        lambda: -1 <= dosage["expression_cn_r"] <= 1,
})

In [ ]:
# YOUR CHECKS - add at least two checks that would catch a real mistake.
# Then use the remaining cells you need for your Option B analysis.

---

# Option C — Does it hold in a different tissue?

**The most straightforward, and available to everybody.**

You tested one cancer type. Is what you found a general property of p53
biology, or something specific to that tissue?

The expression file contains tumours from **twelve** cancer types.
Repeat your TP53 comparison in each and see where your gene behaves the
same way and where it does not.

Two things to be careful about, both marked:

- **Group sizes vary wildly.** TP53 is mutated in most oesophageal
  tumours and very few prostate ones. A comparison with eight mutant
  tumours is not evidence. Set a minimum and say what it is.
- **Twelve tests means twelve chances of a false positive.** At
  p < 0.05, you expect roughly one significant result by luck alone. Say
  something sensible about that — you need not apply a formal
  correction, but you do have to show you know the problem exists.

## The contract

```python
compare_across_cancers(expr, min_group_size)
```

**Takes** `expr`, the tidy expression table, and the minimum number of
mutant **and** wild-type tumours required for a cancer type to be kept.
Reuse the given `compare_groups` function for each eligible cancer.

**Returns** a DataFrame with exactly these columns:

```text
cancer_type, n_mut, n_wt, difference, p_value, significant
```

There should be one row per eligible cancer type, sorted by `p_value`
smallest first, with the index reset. `difference` is mutant minus
wild-type expression.

In [ ]:
def compare_across_cancers(expr, min_group_size):
    # TODO - your turn
    raise NotImplementedError("compare_across_cancers")

In [ ]:
# CHECK - structure and internal consistency
across = compare_across_cancers(expr, 20)
EXPECTED_ACROSS_COLUMNS = ["cancer_type", "n_mut", "n_wt",
                           "difference", "p_value", "significant"]

run_checks({
    "returns a DataFrame with the required columns":
        lambda: (isinstance(across, pd.DataFrame)
                 and list(across.columns) == EXPECTED_ACROSS_COLUMNS),
    "each cancer type appears once":
        lambda: across["cancer_type"].is_unique,
    "every retained group meets the minimum size":
        lambda: (across[["n_mut", "n_wt"]] >= 20).all().all(),
    "rows are sorted by p-value, smallest first":
        lambda: across["p_value"].is_monotonic_increasing,
    "the index was reset":
        lambda: list(across.index) == list(range(len(across))),
    "significant agrees with p < 0.05 in every row":
        lambda: (across["significant"] == (across["p_value"] < 0.05)).all(),
})

In [ ]:
# YOUR CHECKS - add at least two checks that would catch a real mistake.
# Then use the remaining cells you need for your Option C analysis.

---

## Your write-up

Whichever option you chose: what you did, what you found, and — most
importantly — **what it changes about the answer you gave in Notebook
02.**

*Your extension write-up here.*